### RAG Pipeline DATA INGESTION to Vector DB 

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\Kaush\OneDrive\Desktop\Ml\ML Practice\RAC_Prac\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Kaush\OneDrive\Desktop\Ml\ML Practice\RAC_Prac\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf inside the directory
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents=[]
    pdf_dirs=Path(pdf_directory)

    #find allpdf files recursively
    pdf_files=list(pdf_dirs.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} pdf files to be processed")

    for pdf_file in pdf_files:
        print(f"\n Processing:{pdf_file.name}")

        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            #Add source info

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata["file_type"]='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)}pages")
        except Exception as e:
            print(f" Error:{e}")
    print(f"\n total documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")



found 1 pdf files to be processed

 Processing:AJP U3 Notes.pdf
 Loaded 52pages

 total documents loaded:52


In [3]:
### Split the documents into smaller chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks"""
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,
       length_function=len,
       separators=["\n\n", "\n", " ", ""] 
       )
    split_docs=text_splitter.split_documents(documents)
    print(f"split{len(documents)} document into {len(split_docs)} chunks")

    if split_docs:
        print(f"example chunk:\n {split_docs[0].page_content[:200]}...")
        print(f"metadata:{split_docs[0].metadata}")
    return split_docs


In [4]:
chunks=split_documents(all_pdf_documents)

split52 document into 50 chunks
example chunk:
 IntroductiontoJSP
JSPstandsforJavaServerPages. Itisaserver-sidetechnologywhichisusedforcreatingwebapplications.Itisusedtocreatedynamicwebcontent.JSPconsistsofbothHTMLtagsandJSPtags.Inthis,JSPtagsareus...
metadata:{'producer': 'Skia/PDF m125 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'AJP U3 Notes', 'source': '..\\data\\text_files\\AJP U3 Notes.pdf', 'total_pages': 52, 'page': 0, 'page_label': '1', 'source_file': 'AJP U3 Notes.pdf', 'file_type': 'pdf'}


### embedding and vector store db

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        self.model = SentenceTransformer(self.model_name)
        print(f"embedding dimension:{self.model.get_embedding_dimension()}")

    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """Generate embeddings for a list of texts"""
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings for {len(texts)} texts")
        print(f"embedding shape:{embeddings.shape}")
        return embeddings

        ##initialise embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2448.10it/s]


embedding dimension:384


In [10]:
### vector store using chromadb
class VectorStore:
    def __init__(self,collection_name:str="pdf_embeddings",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            
            self.collection=self.client.get_or_create_collection(name=self.collection_name,metadata={"description":"Collection of PDF document embeddings"}
                                                                
            )
            print(f"Initialized vector store with collection name: {self.collection_name}")
            print(f"existing documents in store:{self.collection.count()}")
           
        except Exception as e:

            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """Add documents and their embeddings to the vector store"""
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding ) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(ids=ids,embeddings=embeddings_list,metadatas=metadatas,documents=documents_text)
            print(f"Added {len(documents)} documents to vector store")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise 
vector_store=VectorStore()
vector_store

Initialized vector store with collection name: pdf_embeddings
existing documents in store:0


In [14]:
### convert text to embeddings
texts=[doc.page_content for doc in chunks]

## generate embeddings

embeddings=embedding_manager.generate_embeddings(texts)

### store in vector database

vector_store.add_documents(chunks,embeddings)

Batches: 100%|██████████| 2/2 [00:02<00:00,  1.14s/it]

Generated embeddings for 50 texts
embedding shape:(50, 384)
Added 50 documents to vector store


### RAG Retrieval pipeline

In [ ]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
      

    def retrieve(self,
                 query:str,
                 top_k:int=5,
                 score_threshold:float=0.0)->List[Dict[str,Any]]:
        """Retrieve relevant documents for a given query"""
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]
        try:
            results=self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()],n_results=top_k)

            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):

                    similarity_score=1-distance

                    if similarity_score>=score_threshold:
                     retrieved_docs.append({
                        "id":doc_id,
                        "content":document,
                        "metadata":metadata,
                        "similarity_score":similarity_score,
                        "distance":distance,
                        "rank":i+1
                    })

                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise

rag_retriever=RAGRetriever(vector_store,embedding_manager)